In [2]:
import pandas as pd
import re

df=pd.read_csv("train.csv")

mots_inutiles = set([
    "i","me","my","myself","we","our","ours","ourselves",
    "you","your","yours","yourself","yourselves",
    "he","him","his","himself",
    "she","her","hers","herself",
    "it","its","itself",
    "they","them","their","theirs","themselves",
    "what","which","who","whom","this","that","these","those",
    "am","is","are","was","were","be","been","being",
    "have","has","had","do","does","did",
    "a","an","the","and","or","but","if","because","as",
    "until","while","of","at","by","for","with","about","against",
    "between","into","through","during","before","after","above","below",
    "to","from","up","down","in","out","on","off","over","under",
    "again","further","then","once","get", "just", "like", "can", "not","know",
    "want","time","how","when","now","all","feel", "really", "things", "just",
    "like","very", "too", "lot", "there", "back",
    "even", "every", "years", "told",
    "going", "still", "one", "now","will"
])
contractions = {
    "don't":"do not",
    "can't":"can not",
    "i'm":"i am",
    "it's":"it is",
    "i've":"i have",
    "you're":"you are",
    "we're":"we are",
    "they're":"they are"
}
def preprocess(texte):
    if pd.isna(texte):
        return []

    texte=texte.lower()
    for k,v in contractions.items():
        texte=texte.replace(k, v)
    texte=re.sub(r'[^\w\s]',' ',texte)

    tokens=texte.split()

    tokens=[t for t in tokens if t not in mots_inutiles]
    tokens=[t for t in tokens if len(t) > 2]

    return tokens

df['patient_clean'] = df.iloc[:,0].apply(preprocess)
df['therapist_clean'] = df.iloc[:,1].apply(preprocess)
nb_patients=df.iloc[:,0].nunique()
print("Patients uniques:", nb_patients)
nb_therapeutes=df.iloc[:,1].nunique()
print("Thérapeutes uniques:", nb_therapeutes)
moy_mots_patient = df['patient_clean'].apply(len).mean()
moy_mots_therapist = df['therapist_clean'].apply(len).mean()

print("Mots moyens par patient :", round(moy_mots_patient,2))
print("Mots moyens par thérapeute :", round(moy_mots_therapist,2))

echange_moyens = len(df) / nb_patients
print("Échanges moyens par patient:", echange_moyens)



Patients uniques: 995
Thérapeutes uniques: 2479
Mots moyens par patient : 21.27
Mots moyens par thérapeute : 82.39
Échanges moyens par patient: 3.52964824120603


# **Top 3 mots**

In [4]:
from collections import Counter
mots_patient = [mot for tokens in df['patient_clean'] for mot in tokens]
mots_therapist = [mot for tokens in df['therapist_clean'] for mot in tokens]

freq_patient = Counter(mots_patient)
freq_therapist = Counter(mots_therapist)

print("Top 3 mots patients :", freq_patient.most_common(3))
print("Top 3 mots thérapeutes :", freq_therapist.most_common(3))
tout_mots=[mot for tokens in df['patient_clean'] for mot in tokens]
freq = Counter(tout_mots)
freq_global = Counter(mots_patient + mots_therapist)
print("Top 3 mots global :", freq_global.most_common(3))

Top 3 mots patients : [('never', 623), ('always', 509), ('relationship', 504)]
Top 3 mots thérapeutes : [('may', 2900), ('would', 2367), ('help', 2344)]
Top 3 mots global : [('may', 2921), ('help', 2736), ('would', 2660)]


# **Analyse lexicale**

In [6]:
from nltk.util import ngrams
print("Unigram uniques patients :", len(set(mots_patient)))
print("Unigram uniques thérapeutes :", len(set(mots_therapist)))

def unique_ngrams(data, n):
    ng = []
    for tokens in data:
        ng.extend(list(ngrams(tokens, n)))
    return len(set(ng))

print("Bigram uniques patients :", unique_ngrams(df['patient_clean'], 2))
print("Bigram uniques thérapeutes :", unique_ngrams(df['therapist_clean'], 2))

print("Trigram uniques patients :", unique_ngrams(df['patient_clean'], 3))
print("Trigram uniques thérapeutes :", unique_ngrams(df['therapist_clean'], 3))

Unigram uniques patients : 3465
Unigram uniques thérapeutes : 14020
Bigram uniques patients : 17462
Bigram uniques thérapeutes : 119854
Trigram uniques patients : 18630
Trigram uniques thérapeutes : 154283


In [8]:
pip install sentence-transformers

# **Filtrage via un lexique métier défini manuellement**

In [11]:
themes = {
    "anxiety": ["anxiety","anxious","worry","nervous","panic","fear","stress"],
    "depression": ["depression","sad","hopeless","empty","down"],
    "sleep": ["sleep","insomnia","tired","night","awake"]
}

print("Clés du dictionnaire :", list(themes.keys()))

for k,v in themes.items():
    print(f"clé: {k} taille: {len(v)} -> {v}")
def detect_theme(tokens):
    res = []
    for theme, words in themes.items():
        if any(w in tokens for w in words):
            res.append(theme)
    return res

df['themes_patient'] = df['patient_clean'].apply(detect_theme)



Clés du dictionnaire : ['anxiety', 'depression', 'sleep']
clé: anxiety taille: 7 -> ['anxiety', 'anxious', 'worry', 'nervous', 'panic', 'fear', 'stress']
clé: depression taille: 5 -> ['depression', 'sad', 'hopeless', 'empty', 'down']
clé: sleep taille: 5 -> ['sleep', 'insomnia', 'tired', 'night', 'awake']


# **Méthodes TF-IDF**

In [14]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
import numpy as np

df['patient_text'] = df['patient_clean'].apply(lambda x: " ".join(x))

tfidf = TfidfVectorizer(max_features=2000)
X = tfidf.fit_transform(df['patient_text'])

terms = tfidf.get_feature_names_out()

scores_patient = np.asarray(X.mean(axis=0)).ravel()
top_idx = scores_patient.argsort()[-3:][::-1]
top_words_patient = [terms[i] for i in top_idx]

print("Top 3 TF-IDF patients :", top_words_patient)

df['therapist_text'] = df['therapist_clean'].apply(lambda x: " ".join(x))
X_th = tfidf.fit_transform(df['therapist_text'])

terms_th = tfidf.get_feature_names_out()
scores_th = np.asarray(X_th.mean(axis=0)).ravel()
top_idx_th = scores_th.argsort()[-3:][::-1]

top_words_th = [terms_th[i] for i in top_idx_th]

print("Top 3 TF-IDF thérapeutes :", top_words_th)

global_words = list(set(top_words_patient + top_words_th))
print("Top 3 global TF-IDF :", global_words[:3])

print("Nombre de clusters (K):", 8)

Top 3 TF-IDF patients : ['counseling', 'people', 'should']
Top 3 TF-IDF thérapeutes : ['may', 'would', 'relationship']
Top 3 global TF-IDF : ['relationship', 'would', 'should']
Nombre de clusters (K): 8


# **Clusters patients**

In [16]:
kmeans = KMeans(n_clusters=8, random_state=42)
X = tfidf.fit_transform(df['patient_text'])
kmeans.fit(X)

terms = tfidf.get_feature_names_out()
order_centroids = kmeans.cluster_centers_.argsort()[:, ::-1]

print("\nThèmes des clusters patients :")

for i in range(8):
    top_words = [terms[ind] for ind in order_centroids[i, :5]]
    print(f"Cluster {i} :", top_words)


Thèmes des clusters patients :
Cluster 0 : ['therapist', 'train', 'treatment', 'smoking', 'give']
Cluster 1 : ['therapy', 'normal', 'everytime', 'shaky', 'walk']
Cluster 2 : ['having', 'sex', 'anxiety', 'relationship', 'boyfriend']
Cluster 3 : ['depressed', 'find', 'angry', 'school', 'afraid']
Cluster 4 : ['should', 'something', 'worthless', 'fear', 'mother']
Cluster 5 : ['love', 'wife', 'child', 'says', 'family']
Cluster 6 : ['always', 'talk', 'never', 'sometimes', 'say']
Cluster 7 : ['counseling', 'address', 'history', 'many', 'issues']


In [ ]:
peutes

## **Clusters thérapeutes**

In [17]:
tfidf_th = TfidfVectorizer(max_features=2000)
X_th = tfidf_th.fit_transform(df['therapist_text'])

kmeans_th = KMeans(n_clusters=8, random_state=42)
kmeans_th.fit(X_th)

terms_th = tfidf_th.get_feature_names_out()
order_centroids_th = kmeans_th.cluster_centers_.argsort()[:, ::-1]

print("\nThèmes des clusters thérapeutes :")

for i in range(8):
    top_words = [terms_th[ind] for ind in order_centroids_th[i, :5]]
    print(f"Cluster {i} :", top_words)


Thèmes des clusters thérapeutes :
Cluster 0 : ['drinking', 'smoking', 'problem', 'stop', 'other']
Cluster 1 : ['relationship', 'partner', 'sex', 'trust', 'marriage']
Cluster 2 : ['que', 'para', 'con', 'una', 'las']
Cluster 3 : ['life', 'people', 'more', 'thoughts', 'way']
Cluster 4 : ['child', 'daughter', 'parents', 'parent', 'children']
Cluster 5 : ['anxiety', 'help', 'counseling', 'may', 'depression']
Cluster 6 : ['could', 'would', 'may', 'talk', 'other']
Cluster 7 : ['therapy', 'therapist', 'client', 'cry', 'counselor']


# **Méthode 3: Word Embedding**

##**Thèmes des clusters patients**

In [18]:
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
import numpy as np
embed_model = SentenceTransformer('all-MiniLM-L6-v2')
all_words = list(set(sum(df['patient_clean'], [])))
embeddings = embed_model.encode(all_words)
K = 5
kmeans = KMeans(n_clusters=K, random_state=42)
labels = kmeans.fit_predict(embeddings)

print("Nombre de clusters (K) :", K)
clusters = {i: [] for i in range(K)}

for word, label in zip(all_words, labels):
    clusters[label].append(word)

print("\nThèmes des clusters patients :")
for k, words in clusters.items():
    print(f"Cluster {k} :", words[:10])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Nombre de clusters (K) : 5

Thèmes des clusters patients :
Cluster 0 : ['awkward', 'generally', 'basically', 'different', 'jokes', 'situation', 'reason', 'hell', 'promise', 'wondering']
Cluster 1 : ['contacted', 'discontinued', '3000', 'occurred', 'purchased', 'dies', 'previous', 'plan', 'headed', 'stolen']
Cluster 2 : ['isolated', 'jump', 'breathe', 'rules', 'approach', 'manner', 'improve', 'creative', 'formally', 'stepson']
Cluster 3 : ['crush', 'mall', 'pee', 'book', 'privacy', 'pink', 'social', 'services', 'born', 'occasion']
Cluster 4 : ['prescribed', 'burning', 'courage', 'trauma', 'obsessive', 'having', 'caring', 'guilt', 'psychology', 'ugliness']


## **Thèmes des clusters thérapeutes**

In [21]:
import re
all_words_th = list(sum(df['therapist_clean'], []))
all_words_th = [w for w in all_words_th if w.isalpha() and len(w) > 3]
freq = Counter(all_words_th)
all_words_th = [w for w in all_words_th if freq[w] > 5]
all_words_th = list(set(all_words_th))
embeddings_th = embed_model.encode(all_words_th)
K = 6
kmeans_th = KMeans(n_clusters=K, random_state=42)
labels_th = kmeans_th.fit_predict(embeddings_th)
clusters_th = {i: [] for i in range(K)}

for word, label in zip(all_words_th, labels_th):
    clusters_th[label].append(word)
import numpy as np

print("\nThèmes des clusters thérapeutes (propres) :")

for i in range(K):
    center = kmeans_th.cluster_centers_[i]
    distances = np.linalg.norm(embeddings_th - center, axis=1)
    closest = np.argsort(distances)[:10]

    print(f"\nCluster {i} :")
    print([all_words_th[j] for j in closest])

Nombre de mots conservés : 4311

Thèmes des clusters thérapeutes (propres) :

Cluster 0 :
['explain', 'extremely', 'highly', 'certain', 'explains', 'effectively', 'clarify', 'incredibly', 'almost', 'partly']

Cluster 1 :
['understanding', 'learning', 'actions', 'studied', 'homework', 'task', 'achieve', 'progress', 'observation', 'activity']

Cluster 2 :
['emotion', 'emotionally', 'emotional', 'emotions', 'psychological', 'hurting', 'violence', 'troubled', 'empathy', 'suffering']

Cluster 3 :
['followed', 'responds', 'getting', 'given', 'attempt', 'doing', 'active', 'complete', 'working', 'accept']

Cluster 4 :
['estado', 'esto', 'cuidado', 'escuela', 'diferente', 'contigo', 'fuente', 'siento', 'parece', 'realmente']

Cluster 5 :
['something', 'movie', 'clean', 'body', 'sexually', 'game', 'school', 'name', 'business', 'making']


# **Méthode 4: LDA**

## **Thèmes des topics pour les patients**

In [22]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
n_topics = 5

vectorizer = CountVectorizer(max_features=2000, stop_words='english')

X_patient = vectorizer.fit_transform(df['patient_text'])

lda_patient = LatentDirichletAllocation(
    n_components=n_topics,
    random_state=42
)

lda_patient.fit(X_patient)

words_patient = vectorizer.get_feature_names_out()

for i, topic in enumerate(lda_patient.components_):
    top_words = [words_patient[j] for j in topic.argsort()[-10:]]
    print(f"\nTopic Patient {i}")
    print(top_words)






Topic Patient 0
['sexual', 'family', 'self', 'depression', 'married', 'anxiety', 'address', 'history', 'counseling', 'issues']

Topic Patient 1
['say', 'right', 'boyfriend', 'think', 'girlfriend', 'says', 'relationship', 'past', 'sex', 'love']

Topic Patient 2
['think', 'boyfriend', 'having', 'couple', 'school', 'anxiety', 'therapy', 'normal', 'feeling', 'people']

Topic Patient 3
['bad', 'think', 'sex', 'relationship', 'afraid', 'help', 'doesn', 'thoughts', 'love', 'stop']

Topic Patient 4
['boyfriend', 'make', 'counseling', 'live', 'child', 'life', 'people', 'talk', 'relationship', 'end']


## Thèmes des topics pour les thérapeutes

In [24]:
X_therapist = vectorizer.fit_transform(df['therapist_text'])

lda_therapist = LatentDirichletAllocation(
    n_components=n_topics+1,
    random_state=42
)

lda_therapist.fit(X_therapist)

words_therapist = vectorizer.get_feature_names_out()

for i, topic in enumerate(lda_therapist.components_):
    top_words = [words_therapist[j] for j in topic.argsort()[-10:]]
    print(f"\nTopic Thérapeute {i}")
    print(top_words)


Topic Thérapeute 0
['process', 'depression', 'work', 'issues', 'counselor', 'client', 'counseling', 'therapy', 'help', 'therapist']

Topic Thérapeute 1
['able', 'support', 'ask', 'person', 'try', 'way', 'people', 'good', 'talk', 'help']

Topic Thérapeute 2
['puedes', 'por', 'como', 'las', 'los', 'una', 'sexual', 'para', 'sex', 'que']

Topic Thérapeute 3
['partner', 'good', 'feelings', 'person', 'relationship', 'love', 'way', 'people', 'feeling', 'life']

Topic Thérapeute 4
['stress', 'make', 'thought', 'try', 'way', 'fear', 'help', 'people', 'thoughts', 'anxiety']

Topic Thérapeute 5
['boyfriend', 'family', 'good', 'help', 'person', 'way', 'child', 'make', 'need', 'relationship']
